# 00f - Scraper CNBC Indonesia (korpus pelengkap)

Deskripsi Modul: notebook ini mengumpulkan berita CNBC Indonesia sebagai **korpus pelengkap** (eksperimen tambahan, disetujui asisten).
Desainnya mengikuti `00a_scraper`: sampling per interval hari, batas artikel per interval, penyaring kata kunci, checkpoint, dan multi-thread.

Perbedaan utama dari `00a_scraper`:
* Artikel ditemukan lewat halaman indeks CNBC Indonesia (`/indeks?page=N`), bukan Google News RSS, sehingga daftar lengkap dan waktu terbit tepat sampai detik (diambil dari URL).
* Kata kunci dalam bahasa Indonesia. **Daftar kata kunci di bawah hanyalah titik awal: kelompok perlu menyunting sendiri.**
* Laporan harga harian (misal "Pukul 13.00 WIB: Rupiah Melemah ke Rp 14.260/US$") dibuang karena memuat jawaban (kebocoran label).
* Hanya 3 worker dan jeda 1,5 detik per permintaan agar tidak membebani situs.

## 0. Library

In [21]:
%pip install -q requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Parameter
Seluruh parameter dapat diganti lewat environment variable agar keluaran mentah dapat disimpan di luar repo.

In [22]:
import os, re, time, random, threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import pandas as pd

START_DATE = datetime.fromisoformat(os.environ.get("CNBCID_START", "2021-09-01"))
END_DATE = datetime.fromisoformat(os.environ.get("CNBCID_END", "2026-08-31"))
SAMPLE_INTERVAL_DAYS = int(os.environ.get("CNBCID_INTERVAL", "3"))   # 1 hari dicuplik tiap interval
MAX_PER_INTERVAL = int(os.environ.get("CNBCID_MAXPER", "40"))
WORKERS = int(os.environ.get("CNBCID_WORKERS", "3"))
DELAY = float(os.environ.get("CNBCID_DELAY", "1.5"))
SEED = 42

OUTPUT_DIR = os.environ.get("CNBCID_OUT", os.path.join("..", "data", "raw", "cnbcindonesia"))
os.makedirs(OUTPUT_DIR, exist_ok=True)
SAVE_PATH = os.path.join(OUTPUT_DIR, "cnbcindonesia_news.csv")
CHECKPOINT = os.path.join(OUTPUT_DIR, "completed_days.txt")

BASE = "https://www.cnbcindonesia.com"
KEEP_SECTIONS = {"news", "market", "mymoney"}
MIN_INDEX = 30                # indeks harian yang terbaca kurang dari ini dianggap gagal (biasanya ~250)
MAX_FAIL_FRAC = 0.3           # hari dianggap gagal jika >30% unduhan artikel gagal
PAGES_PER_DAY = 21           # perkiraan awal, dikoreksi otomatis saat mencari halaman
MAX_PAGE = 61875
COOLDOWN_MIN = float(os.environ.get("CNBCID_COOLDOWN_MIN", "10"))   # menit tunggu pertama setelah dibatasi (berlipat dua tiap kali)
MAX_COOLDOWNS = int(os.environ.get("CNBCID_MAX_COOLDOWNS", "6"))      # berapa kali menunggu sebelum menyerah
EMPTY_LIMIT = int(os.environ.get("CNBCID_EMPTY_LIMIT", "6"))   # hari gagal beruntun (indeks kosong) sebelum berhenti


## 2. Kata kunci (SUNTING SENDIRI)
* `CORE_KEYWORDS`: artikel hanya disimpan jika judul (dan paragraf awal) memuat salah satunya. Setiap entri berupa regex dan dicocokkan sebagai kata utuh (mis. `perang` tidak lagi cocok dengan `perangkat`).
* `LEAK_PATTERNS`: laporan harga/kurs yang dibuang agar fitur teks tidak memuat jawaban.

In [23]:
CORE_KEYWORDS = [
    # konflik dan militer
    r"perang", r"invasi", r"rudal", r"kudeta", r"tentara", r"nato", r"gencatan senjata", r"geopolitik\w*",
    r"serangan (?:rudal|udara|militer|drone)",
    r"konflik (?:timur tengah|rusia|ukraina|israel|palestina|laut china selatan)",
    # negara dan tokoh
    r"rusia", r"ukraina", r"putin", r"zelensky", r"israel", r"gaza", r"hamas", r"hizbullah", r"houthi",
    r"netanyahu", r"iran", r"taiwan", r"korea utara", r"xi jinping", r"laut china selatan",
    r"trump", r"biden", r"gedung putih",
    # perdagangan dan sanksi
    r"sanksi (?:ekonomi|barat|as|amerika|eropa|baru)", r"embargo", r"perang dagang", r"tarif (?:impor|dagang|trump|resiprokal|balasan)", r"bea masuk",
    r"pembatasan ekspor",
    # bank sentral dan makro global
    r"the fed", r"federal reserve", r"fomc", r"powell", r"ecb", r"bank of japan", r"bank sentral",
    r"suku bunga(?: acuan)?", r"bi rate", r"bank indonesia", r"inflasi", r"resesi", r"stagflasi",
    r"krisis (?:energi|pangan|utang|keuangan|perbankan|ekonomi global)", r"gagal bayar utang",
    r"imf", r"bank dunia", r"g20", r"g7", r"brics", r"opec\+?", r"minyak mentah", r"brent", r"wti",
]
LEAK_PATTERNS = [
    r"rupiah", r"\bkurs", r"\bidr\b", r"us\$", r"dolar as", r"\busd\b", r"pukul \d", r"sesi (?:i|ii|1|2)\b",
    r"ihsg", r"jisdor", r"\brp ?\d",
]
# Kata yang ambigu (sering muncul di berita non-geopolitik) hanya dihitung jika muncul bersama kata konteks.
AMBIGUOUS_KEYWORDS = [r"china", r"tiongkok"]
CONTEXT_KEYWORDS = [
    r"dagang", r"ekspor", r"impor", r"tarif", r"ekonomi", r"pertumbuhan", r"pasokan", r"energi", r"minyak",
    r"yuan", r"evergrande", r"properti", r"utang", r"langka", r"logam",
    r"as", r"amerika", r"biden", r"trump", r"taiwan", r"rusia", r"sanksi", r"nato", r"natuna",
    r"laut china selatan", r"militer", r"nuklir", r"rudal",
    r"bank sentral", r"pboc", r"suku bunga", r"regulasi",
]
CORE_RE = re.compile(r"\b(?:" + "|".join(CORE_KEYWORDS) + r")\b", re.I)
AMBIGUOUS_RE = re.compile(r"\b(?:" + "|".join(AMBIGUOUS_KEYWORDS) + r")\b", re.I)
CONTEXT_RE = re.compile(r"\b(?:" + "|".join(CONTEXT_KEYWORDS) + r")\b", re.I)


def is_relevant(text):
    """Relevan jika memuat kata inti, atau kata ambigu (china/tiongkok) bersama kata konteks."""
    return bool(CORE_RE.search(text)) or bool(AMBIGUOUS_RE.search(text) and CONTEXT_RE.search(text))
LEAK_RE = re.compile("|".join(LEAK_PATTERNS), re.I)

## 3. Fungsi pembantu: HTTP, halaman indeks, dan isi artikel

In [24]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (UGM NLP coursework, low-rate crawler)",
    "Accept-Language": "id,en;q=0.8",
})

URL_RE = re.compile(r'https://www\.cnbcindonesia\.com/([a-z\-]+)/(\d{14})-(\d+)-(\d+)/([^"\'\s<>?#]*)')


class BlockedError(Exception):
    """Situs tampaknya memblokir/membatasi permintaan; proses dihentikan agar tidak membebani situs."""


STOP = threading.Event()
BLOCK_LIMIT = int(os.environ.get("CNBCID_BLOCK_LIMIT", "8"))   # 403/429 beruntun sebelum berhenti
_blocked = 0
_stop_reason = "dihentikan"
_blk_lock = threading.Lock()


def fetch(url, tries=5):
    """GET dengan jeda sopan dan backoff untuk 403/429/5xx.
    Berhenti total (BlockedError) jika 403/429 terjadi beruntun BLOCK_LIMIT kali."""
    global _blocked, _stop_reason
    for k in range(tries):
        if STOP.is_set():
            raise BlockedError(_stop_reason)
        time.sleep(DELAY)
        try:
            r = session.get(url, timeout=30)
        except requests.RequestException as e:
            print("  jaringan:", type(e).__name__, url[-40:])
            time.sleep(5 * (k + 1))
            continue
        if r.status_code == 200:
            with _blk_lock:
                _blocked = 0
            return r.text
        if r.status_code in (403, 429):
            with _blk_lock:
                _blocked += 1
                if _blocked >= BLOCK_LIMIT:
                    _stop_reason = f"{_blocked} respons {r.status_code} beruntun"
                    STOP.set()
                    raise BlockedError(_stop_reason)
            time.sleep(15 * (k + 1))
            continue
        if r.status_code in (500, 502, 503, 504):
            time.sleep(10 * (k + 1))
            continue
        print("  status", r.status_code, url[-40:])
        return None          # 404 dll: tidak dicoba ulang
    return None


def parse_index(html):
    """Daftar artikel di satu halaman indeks: (waktu, bagian, url, slug)."""
    seen, out = set(), []
    for m in URL_RE.finditer(html):
        sec, ts, _a, _b, slug = m.groups()
        url = m.group(0)
        if url in seen:
            continue
        seen.add(url)
        out.append((datetime.strptime(ts, "%Y%m%d%H%M%S"), sec, url, slug))
    return out


def index_page(p):
    html = fetch(f"{BASE}/indeks?page={p}")
    return parse_index(html) if html else []


def crawl_day(day):
    """Kumpulkan seluruh artikel pada satu hari (WIB) dari halaman indeks yang berurutan terbaru-ke-lama."""
    lo, hi = day, day + timedelta(days=1)
    p = max(1, min(MAX_PAGE, int((datetime.now() - hi).total_seconds() / 86400 * PAGES_PER_DAY)))
    items = []
    # 1) cari halaman yang mulai memuat artikel yang lebih baru dari batas atas hari tersebut
    for _ in range(14):
        it = index_page(p)
        if not it:
            return None
        mx, mn = max(t for t, *_ in it), min(t for t, *_ in it)
        if mx < hi and p > 1:                 # halaman terlalu tua -> naik (halaman lebih kecil)
            p = max(1, p - max(1, int((hi - mx).total_seconds() / 86400 * PAGES_PER_DAY)))
        elif mn > hi:                          # halaman terlalu baru -> turun
            p = min(MAX_PAGE, p + max(1, int((mn - hi).total_seconds() / 86400 * PAGES_PER_DAY)))
        else:
            break
    # 2) mundur satu halaman sebagai pengaman, lalu telusuri ke halaman yang lebih tua
    p = max(1, p - 1)
    while p <= MAX_PAGE:
        it = index_page(p)
        if not it:
            break
        items += [x for x in it if lo <= x[0] < hi]
        if max(t for t, *_ in it) < lo:
            break
        p += 1
    uniq = {x[2]: x for x in items}
    return list(uniq.values())


def prefilter(items):
    """Saring awal dari slug URL (belum mengunduh isi artikel)."""
    keep = []
    for ts, sec, url, slug in items:
        s = slug.replace("-", " ")
        if sec in KEEP_SECTIONS and not LEAK_RE.search(s) and is_relevant(s):
            keep.append((ts, sec, url, slug))
    return keep


def parse_article(html):
    import html as H
    t = re.search(r"<title>(.*?)</title>", html, re.S)
    title = H.unescape(t.group(1)).strip() if t else ""
    title = re.sub(r"\s*[-|]\s*CNBC Indonesia\s*$", "", title)
    i = html.find('<div class="detail-text')
    if i < 0:
        return title, "", ""
    # Iklan disisipkan di tengah artikel sebagai <div>, sehingga regex "sampai </div> pertama" memotong isi.
    # Ambil seluruh <p> sampai penanda akhir artikel (tautan "Next Article" atau </article>).
    seg = html[i:]
    ends = [k for k in (seg.find("Next Article"), seg.find("</article>")) if k > 0]
    seg = seg[:min(ends)] if ends else seg
    paras = [re.sub(r"<[^>]+>", "", H.unescape(p)).strip() for p in re.findall(r"<p[^>]*>(.*?)</p>", seg, re.S)]
    paras = [p for p in paras if p]
    if not paras:
        return title, "", ""
    paras[0] = re.sub(r"^[A-Za-z .]+,\s*CNBC Indonesia\s*-\s*", "", paras[0])
    return title, paras[0], " ".join(paras)


def passes_final_filter(title, lead, text):
    head = f"{title} {lead}"
    return len(text.split()) >= 30 and is_relevant(head) and LEAK_RE.search(head) is None


def process_day(day):
    items = crawl_day(day)
    if items is None:
        return None
    pool = prefilter(items)
    rng = random.Random(SEED + int(day.strftime("%Y%m%d")))
    if len(pool) > MAX_PER_INTERVAL:
        pool = rng.sample(pool, MAX_PER_INTERVAL)
    rows, n_fail = [], 0
    for ts, sec, url, slug in pool:
        html = fetch(url)
        if not html:
            n_fail += 1
            continue
        title, lead, text = parse_article(html)
        if passes_final_filter(title, lead, text):
            rows.append(dict(published_at=ts, date=ts.date(), section=sec, title=title,
                             lead_paragraph=lead, content=text, word_count=len(text.split()), url=url))
    # hari dianggap sah hanya jika indeks terbaca wajar dan unduhan artikel tidak banyak gagal
    ok = len(items) >= MIN_INDEX and n_fail <= MAX_FAIL_FRAC * max(1, len(pool))
    return dict(day=day, n_index=len(items), n_prefilter=len(pool), n_fail=n_fail, rows=rows, ok=ok)

## 4. Checkpoint dan pelaksanaan
Setiap hari yang selesai langsung ditulis ke CSV dan dicatat di `completed_days.txt`, sehingga proses dapat dilanjutkan jika terhenti.

In [25]:
days, d = [], START_DATE
while d <= END_DATE:
    days.append(d)
    d += timedelta(days=SAMPLE_INTERVAL_DAYS)
done = set(open(CHECKPOINT).read().split()) if os.path.exists(CHECKPOINT) else set()
pending = [x for x in days if x.strftime("%Y-%m-%d") not in done]
print(f"{len(days)} hari cuplikan, {len(done)} selesai, {len(pending)} tersisa")

lock = threading.Lock()


def save(res):
    """Tulis baris dan checkpoint bersamaan, hanya untuk hari yang sah."""
    with lock:
        if res["rows"]:
            pd.DataFrame(res["rows"]).to_csv(SAVE_PATH, mode="a", index=False, header=not os.path.exists(SAVE_PATH))
        with open(CHECKPOINT, "a") as f:
            f.write(res["day"].strftime("%Y-%m-%d") + "\n")


def run_pass(todo, label):
    """Satu putaran: kembalikan daftar hari yang gagal. Berhenti total jika situs memblokir."""
    failed, n_done, n_rows, t0, n_bad = [], 0, 0, time.time(), 0
    ex = ThreadPoolExecutor(max_workers=WORKERS)
    futs = {ex.submit(process_day, x): x for x in todo}
    try:
        for fu in as_completed(futs):
            day = futs[fu]
            try:
                res = fu.result()
            except BlockedError:
                raise
            except Exception as e:
                print("ERR", day.date(), repr(e)[:120])
                failed.append(day)
                continue
            if res is None or not res["ok"]:
                info = "" if res is None else f"(indeks={res['n_index']}, unduhan gagal={res['n_fail']}/{res['n_prefilter']})"
                print("gagal, tidak dicheckpoint", day.date(), info)
                failed.append(day)
                n_bad += 1
                if n_bad >= EMPTY_LIMIT:
                    raise BlockedError(f"{n_bad} hari gagal beruntun (indeks kosong/tidak terbaca)")
                continue
            save(res)
            n_bad = 0
            n_done += 1
            n_rows += len(res["rows"])
            if n_done % 5 == 0:
                print(f"[{label} {n_done}/{len(todo)}] {day.date()} indeks={res['n_index']} disimpan={len(res['rows'])} "
                      f"gagal_unduh={res['n_fail']} total_baris={n_rows} {(time.time()-t0)/60:.1f} menit")
    except BlockedError as e:
        STOP.set()
        for f in futs:
            f.cancel()
        print(f"\nDIBATASI: {e}.")
        ex.shutdown(wait=True, cancel_futures=True)   # fetch berhenti cepat karena STOP menyala
        raise
    ex.shutdown(wait=True)
    return failed


def pending_days():
    done_now = set(open(CHECKPOINT).read().split()) if os.path.exists(CHECKPOINT) else set()
    return [x for x in days if x.strftime("%Y-%m-%d") not in done_now]


def crawl_with_cooldown():
    """Jika dibatasi situs: tunggu (10, 20, 40, ... menit), lalu lanjutkan dari checkpoint."""
    global _blocked, _stop_reason
    for attempt in range(MAX_COOLDOWNS + 1):
        todo = pending_days()
        if not todo:
            return []
        try:
            failed = run_pass(todo, f"putaran {attempt + 1}a")
            if failed:
                print(f"\nPutaran ulang untuk {len(failed)} hari yang gagal ...")
                failed = run_pass(failed, f"putaran {attempt + 1}b")
            return failed
        except BlockedError:
            if attempt == MAX_COOLDOWNS:
                print(f"Menyerah setelah {MAX_COOLDOWNS} kali menunggu. Jalankan ulang sel ini nanti (progres tersimpan).")
                return pending_days()
            wait_min = COOLDOWN_MIN * 2 ** attempt
            print(f"Menunggu {wait_min:.0f} menit sebelum mencoba lagi (percobaan {attempt + 1}/{MAX_COOLDOWNS}) ...")
            time.sleep(wait_min * 60)
            with _blk_lock:
                _blocked = 0
            _stop_reason = "dihentikan"
            STOP.clear()


failed = crawl_with_cooldown()
print("selesai." if not failed else f"selesai; {len(failed)} hari belum selesai, jalankan ulang sel ini nanti: {[d.date().isoformat() for d in failed][:10]} ...")


609 hari cuplikan, 609 selesai, 0 tersisa
selesai.
